In [ ]:
from ts2.data.cell_dataset import CellBenchDataset
import torch
import pandas as pd
import altair as alt

In [ ]:
cbd = CellBenchDataset(
                 data_root="/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/scbench/",
                 slides_file= "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/data/scbench/mosaic.csv",
                 transform= lambda x: torch.stack(((x/2**16).mean(dim=[-1,-2]), (x/2**16).std(dim=[-1,-2]))),
                 balance_instance_class=False,
                 num_sample_wo_replacement=None,
                 class_filter= ['glial/astrocytes', 'glial/neurons', 'glial/oligodendrocytes', 'glial/reactive_astrocytes', 'glioma/giant_cell', 'glioma/tumor_cells', 'immune/lymphocytes', 'immune/macrophages', 'metastatic/adenocarcinoma', 'metastatic/melanoma', 'metastatic/sarcoma', 'metastatic/squamous_cell', 'proliferation/mitotic_figures']
)

In [ ]:
im_stats = torch.stack([cbd.__getitem__(i)["image"].flatten() for i in range(len(cbd))])
im_label = [i["label"] for i in cbd.instances_]
im_path = [i["path"] for i in cbd.instances_]
ims = pd.DataFrame({"ch2_mean": im_stats[:,0],
                    "ch3_mean": im_stats[:,1],
                    "ch2_std": im_stats[:,2],
                    "ch3_std": im_stats[:,3],
                    "im_label": im_label,
                    "im_path": im_path})
ims["slide"] = ims["im_path"].str.extract("(NIO_[A-Z]+_[0-9]+[a-zA-Z]?.[0-9]+[a-zA-Z]?)")

In [ ]:
slide_metric = ims.drop("im_path", axis=1).groupby(["slide", "im_label"]).agg(["mean", "std"])
slide_metric.columns = ['_'.join(col).strip("_") for col in slide_metric.columns.values]
slide_metric = slide_metric.reset_index().fillna(0)
slide_metric["max_mean_std"] = slide_metric[["ch2_mean_std", "ch3_mean_std"]].max(axis=1)

In [ ]:
noslide_metric = ims.drop(["im_path", "slide"], axis=1).groupby(["im_label"]).agg(["mean", "std"])
noslide_metric.columns = ['_'.join(col).strip("_") for col in noslide_metric.columns.values]
noslide_metric = noslide_metric.reset_index().fillna(0)
noslide_metric["max_mean_std"] = noslide_metric[["ch2_mean_std", "ch3_mean_std"]].max(axis=1)

In [ ]:
alt.data_transformers.disable_max_rows()
class_selection = alt.selection_point(fields=["im_label"], bind='legend')

slide_list = sorted(set(slide_metric["slide"].tolist()))
slide_bind = alt.binding_select(options=[None] + slide_list,
                                labels=["All"] + slide_list,
                                name='Slide ')
slide_selection = alt.selection_point(fields=['slide'], bind=slide_bind)

slide_chart = alt.Chart(slide_metric).mark_point(shape="circle", filled=False).encode(
    x=alt.X(
        "ch2_mean_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    y=alt.X(
        "ch3_mean_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    #size="max_mean_std",
    color="im_label",
    tooltip=["slide", "im_label", "ch2_mean_std", "ch3_mean_std"],
    opacity=alt.condition(
        class_selection & slide_selection,
        alt.value(1.0), alt.value(0.1)
    )
).add_params(
    class_selection,
    slide_selection
)

noslide_chart = alt.Chart(noslide_metric).mark_point(shape="circle", filled=True).encode(
    x=alt.X(
        "ch2_mean_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    y=alt.X(
        "ch3_mean_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    #size="max_mean_std",
    color="im_label",
    tooltip=["im_label", "ch2_mean_std", "ch3_mean_std"],
    opacity=alt.condition(
        class_selection,
        alt.value(1.0), alt.value(0.1)
    )
).add_params(
    class_selection,
)



(slide_chart+noslide_chart).properties(
    width=700,
    height=700
).interactive()

# This is mean and std of average pixel value

In [ ]:
cbd = CellBenchDataset(
                 data_root="/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/scbench/",
                 slides_file= "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/data/scbench/mosaic.csv",
                 transform= lambda x: x/(2**16),
                 balance_instance_class=False,
                 num_sample_wo_replacement=None,
                 class_filter= ['glial/astrocytes', 'glial/neurons', 'glial/oligodendrocytes', 'glial/reactive_astrocytes', 'glioma/giant_cell', 'glioma/tumor_cells', 'immune/lymphocytes', 'immune/macrophages', 'metastatic/adenocarcinoma', 'metastatic/melanoma', 'metastatic/sarcoma', 'metastatic/squamous_cell', 'proliferation/mitotic_figures']
)

In [ ]:
im_stats = [cbd.__getitem__(i)["image"].flatten(start_dim=-2, end_dim=-1) for i in range(len(cbd))]
im_label = [i["label"] for i in cbd.instances_]
im_path = [i["path"] for i in cbd.instances_]
ims = pd.DataFrame({"im": im_stats,
                    "im_label": im_label,
                    "im_path": im_path})
ims["slide"] = ims["im_path"].str.extract("(NIO_[A-Z]+_[0-9]+[a-zA-Z]?.[0-9]+[a-zA-Z]?)")

In [ ]:
slide_metric = ims.drop("im_path", axis=1).groupby(["slide", "im_label"]).agg([
    lambda x: torch.cat(x.tolist())[:,0].mean().item(),
    lambda x: torch.cat(x.tolist())[:,1].mean().item(),
    lambda x: torch.cat(x.tolist())[:,0].std().item(),
    lambda x: torch.cat(x.tolist())[:,1].std().item()
])
slide_metric.columns = ["ch2_mean", "ch3_mean", "ch2_std", "ch3_std"]
slide_metric = slide_metric.reset_index().fillna(0)
slide_metric["max_std"] = slide_metric[["ch2_std", "ch3_std"]].max(axis=1)

In [ ]:
noslide_metric = ims.drop(["im_path", "slide"], axis=1).groupby(["im_label"]).agg([
    lambda x: torch.cat(x.tolist())[:,0].mean().item(),
    lambda x: torch.cat(x.tolist())[:,1].mean().item(),
    lambda x: torch.cat(x.tolist())[:,0].std().item(),
    lambda x: torch.cat(x.tolist())[:,1].std().item()
])
noslide_metric.columns = ["ch2_mean", "ch3_mean", "ch2_std", "ch3_std"]
noslide_metric = noslide_metric.reset_index().fillna(0)
noslide_metric["max_std"] = noslide_metric[["ch2_std", "ch3_std"]].max(axis=1)

In [ ]:
alt.data_transformers.disable_max_rows()
class_selection = alt.selection_point(fields=["im_label"], bind='legend')

slide_list = sorted(set(slide_metric["slide"].tolist()))
slide_bind = alt.binding_select(options=[None] + slide_list,
                                labels=["All"] + slide_list,
                                name='Slide ')
slide_selection = alt.selection_point(fields=['slide'], bind=slide_bind)

slide_chart = alt.Chart(slide_metric).mark_point(shape="circle", filled=False).encode(
    x=alt.X(
        "ch2_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    y=alt.X(
        "ch3_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    #size="max_std",
    color="im_label",
    tooltip=["slide", "im_label", "ch2_std", "ch3_std"],
    opacity=alt.condition(
        class_selection & slide_selection,
        alt.value(1.0), alt.value(0.1)
    )
).add_params(
    class_selection,
    slide_selection
)

noslide_chart = alt.Chart(noslide_metric).mark_point(shape="circle", filled=True).encode(
    x=alt.X(
        "ch2_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    y=alt.X(
        "ch3_mean",
        axis=alt.Axis(tickSize=0),
        scale=alt.Scale(domain=[0.1,0.35])
    ),
    #size=alt.Size("max_std"),
    color="im_label",
    tooltip=["im_label", "ch2_std", "ch3_std"],
    opacity=alt.condition(
        class_selection,
        alt.value(1.0), alt.value(0.1)
    )
).add_params(
    class_selection,
)



(slide_chart+noslide_chart).properties(
    width=700,
    height=700
).interactive()

# This is mean and std of average pixel value

In [ ]:
from ts2.data.transforms import HistologyTransform
from ts2.data.db_improc import instantiate_process_read
from ts2.data.meta_parser import CachedCSVParser

In [ ]:
transform_ = HistologyTransform(
    which_set="scsrh",
    base_aug_params={
                "laser_noise_config": None,
                "get_third_channel_params": {
                    "mode": "three_channels",
                    "subtracted_base":0.07629394531
                },
                "mask_params":{
                    "how_to_process": "small_patch"
                },
                "to_uint8": False
            },
    strong_aug_params={
        "aug_list": [
            {
                "which": "center_crop_always_apply",
                "params":{
                  "size": 48
                }
            }
        ],
        "aug_prob": 0
    }
)

In [ ]:
cbd = CellBenchDataset(
     data_root="/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/scbench/",
     slides_file= "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/data/scbench/mosaic.csv",
     transform= transform_,
     balance_instance_class=False,
     num_sample_wo_replacement=None,
     class_filter= ['glial/astrocytes', 'glial/neurons', 'glial/oligodendrocytes', 'glial/reactive_astrocytes', 'glioma/giant_cell', 'glioma/tumor_cells', 'immune/lymphocytes', 'immune/macrophages', 'metastatic/adenocarcinoma', 'metastatic/melanoma', 'metastatic/sarcoma', 'metastatic/squamous_cell', 'proliferation/mitotic_figures']
)

In [ ]:
images = torch.cat([cbd.__getitem__(i)["image"] for i in range(len(cbd))])

In [ ]:
images.shape

In [ ]:
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
import einops
import matplotlib.pyplot as plt
def srh_to_uint8_func(x):
    return einops.rearrange((adjust_brightness(adjust_contrast(x, 2), 3) * 255).to(torch.uint8), "c h w -> h w c")

In [ ]:
images.mean(dim=[0,-1,-2])

In [ ]:
images.std(dim=[0,-1,-2])

In [ ]:
mean = torch.tensor([0.0872, 0.1546, 0.1604]).unsqueeze(-1).unsqueeze(-1)
std = torch.tensor([0.0444, 0.0939, 0.0560]).unsqueeze(-1).unsqueeze(-1)

In [ ]:
instance_norm = torch.nn.InstanceNorm2d(num_features=3)

In [ ]:
import numpy as np
np.random.permutation(len(images))

In [ ]:
for i in np.random.permutation(len(images))[:50]:
    fig, ax = plt.subplots(1,2)
    ax[0].imshow(srh_to_uint8_func(images[i]))
    ax[1].imshow(srh_to_uint8_func((instance_norm(images[i]) * std + mean).clamp(0, 1)))
